In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/datasets/konstantineendeladze/basic-clean-done-dataset/test_clean.parquet
/kaggle/input/datasets/konstantineendeladze/basic-clean-done-dataset/y.csv
/kaggle/input/datasets/konstantineendeladze/basic-clean-done-dataset/X_clean.parquet


In [20]:
import pandas as pd
import os

# Base path
path = "/kaggle/input/datasets/konstantineendeladze/basic-clean-done-dataset/"

# Load data
X_clean = pd.read_parquet(os.path.join(path, "X_clean.parquet"))
test_clean = pd.read_parquet(os.path.join(path, "test_clean.parquet"))
y = pd.read_csv(os.path.join(path, "y.csv"))

# ---- Sanity checks ----
print("=== DATA LOADED SUCCESSFULLY ===")

print("\nTrain shape (X_clean):", X_clean.shape)
print("Test shape (test_clean):", test_clean.shape)
print("Target shape (y):", y.shape)

print("\nX_clean columns (first 10):")
print(X_clean.columns[:10])

print("\ny head:")
print(y.head())

print("\nMissing values in X_clean:")
print(X_clean.isna().sum().sort_values(ascending=False).head(10))

print("\nDtypes overview:")
print(X_clean.dtypes.value_counts())

=== DATA LOADED SUCCESSFULLY ===

Train shape (X_clean): (590540, 424)
Test shape (test_clean): (590540, 424)
Target shape (y): (590540, 1)

X_clean columns (first 10):
Index(['TransactionID', 'TransactionDT', 'TransactionAmt', 'ProductCD',
       'card1', 'card2', 'card3', 'card4', 'card5', 'card6'],
      dtype='object')

y head:
   isFraud
0        0
1        0
2        0
3        0
4        0

Missing values in X_clean:
dist2    552913
D7       551623
id_18    545427
D13      528588
D14      528353
D12      525823
id_04    524216
id_03    524216
D6       517353
id_33    517251
dtype: int64

Dtypes overview:
float32    392
object      29
int32        2
int16        1
Name: count, dtype: int64


**Cleaning**

In [21]:
y_target = y["isFraud"]
X = X_clean.copy()

print("Target distribution:")
print(y_target.value_counts(normalize=True))

Target distribution:
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [22]:
drop_cols = ["TransactionID"]
X = X.drop(columns=[c for c in drop_cols if c in X.columns])

print("Dropped columns:", drop_cols)
print("New shape:", X.shape)

Dropped columns: ['TransactionID']
New shape: (590540, 423)


In [23]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", len(cat_cols))
print("Numeric columns:", len(num_cols))

Categorical columns: 29
Numeric columns: 394


In [24]:
missing_ratio = X.isna().mean().sort_values(ascending=False)

print("Top missing features:")
print(missing_ratio.head(15))

Top missing features:
dist2    0.936284
D7       0.934099
id_18    0.923607
D13      0.895093
D14      0.894695
D12      0.890410
id_03    0.887689
id_04    0.887689
D6       0.876068
id_33    0.875895
D9       0.873123
id_09    0.873123
D8       0.873123
id_10    0.873123
id_30    0.868654
dtype: float64


**Feature Engineering**

In [25]:
import numpy as np

X_fe = X.copy()

# 1. Transaction time features (common in fraud datasets)
if "TransactionDT" in X_fe.columns:
    X_fe["TransactionDT_day"] = X_fe["TransactionDT"] // (60 * 60 * 24)
    X_fe["TransactionDT_hour"] = (X_fe["TransactionDT"] // 3600) % 24
    X_fe = X_fe.drop(columns=["TransactionDT"])

# 2. Optional: log transform for skewed monetary feature
if "TransactionAmt" in X_fe.columns:
    X_fe["TransactionAmt_log"] = np.log1p(X_fe["TransactionAmt"])

print("Feature engineering complete")
print("New shape:", X_fe.shape)
print("Sample columns:", X_fe.columns[:10])

Feature engineering complete
New shape: (590540, 425)
Sample columns: Index(['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4',
       'card5', 'card6', 'addr1', 'addr2'],
      dtype='object')


**Preprocessing Pipeline**

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# 1. Split target
y_target = y["isFraud"]

# 2. Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_fe, y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target
)

# 3. Column types
cat_cols = X_train.select_dtypes(include=["object"]).columns
num_cols = X_train.select_dtypes(exclude=["object"]).columns

# 4. Numeric pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 5. Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 6. Combine
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

print("Preprocessing pipeline created")
print("Numeric features:", len(num_cols))
print("Categorical features:", len(cat_cols))

Preprocessing pipeline created
Numeric features: 396
Categorical features: 29


In [27]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib
import os

# 1. Define model
log_reg = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    class_weight="balanced"
)

# 2. Full pipeline
clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", log_reg)
])

# 3. Train
clf.fit(X_train, y_train)

# 4. IMMEDIATE SAVE (add this)
save_path = "/kaggle/working/logreg_fraud_model.pkl"
joblib.dump(clf, save_path)

print(f"Model saved at: {save_path}")

# 5. Predictions
y_pred_proba = clf.predict_proba(X_val)[:, 1]
y_pred = clf.predict(X_val)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model saved at: /kaggle/working/logreg_fraud_model.pkl


In [28]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Predictions (in case kernel reset or variables reused)
y_pred_proba = clf.predict_proba(X_val)[:, 1]
y_pred = clf.predict(X_val)

# Metrics
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_pred_proba)

print("=== LOGISTIC REGRESSION RESULTS ===")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", auc)

# Confusion matrix (very important for fraud)
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

=== LOGISTIC REGRESSION RESULTS ===
Precision: 0.15445583791549017
Recall: 0.7544156786837648
F1: 0.25641447368421055
ROC-AUC: 0.8838745661567532

Confusion Matrix:
[[96906 17069]
 [ 1015  3118]]


In [29]:
import numpy as np
import pandas as pd

# Get probabilities
y_probs = clf.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)

results = []

for t in thresholds:
    y_pred_t = (y_probs >= t).astype(int)
    
    precision = precision_score(y_val, y_pred_t)
    recall = recall_score(y_val, y_pred_t)
    f1 = f1_score(y_val, y_pred_t)
    
    results.append([t, precision, recall, f1])

results_df = pd.DataFrame(results, columns=["threshold", "precision", "recall", "f1"])

# Best threshold by F1
best_row = results_df.loc[results_df["f1"].idxmax()]

print("=== BEST THRESHOLD (by F1) ===")
print(best_row)

best_threshold = best_row["threshold"]

print("\nBest threshold:", best_threshold)

=== BEST THRESHOLD (by F1) ===
threshold    0.871212
precision    0.527479
recall       0.438906
f1           0.479134
Name: 87, dtype: float64

Best threshold: 0.8712121212121212


In [30]:
import dagshub
import mlflow
import os
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

dagshub.init(
    repo_owner="kende23",
    repo_name="ieee-cis-fraud-detection",
    mlflow=True
)

mlflow.set_experiment("fraud-logreg-baseline")

# credentials (must be BEFORE logging)
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2fb81bc6-bb6a-48fc-905d-6684eff35682&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e797cdcf25207a454d1da648a04950cc584013a16fe351c6e3f8d80fd5185d8d




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

2026/05/01 20:24:15 INFO mlflow.tracking.fluent: Experiment with name 'fraud-logreg-baseline' does not exist. Creating a new experiment.


In [31]:
y_probs = clf.predict_proba(X_val)[:, 1]
y_pred = (y_probs >= best_threshold).astype(int)

precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_probs)

In [32]:
with mlflow.start_run():

    # log model
    mlflow.sklearn.log_model(clf, "model")

    # metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("threshold", float(best_threshold))

    # save locally

    print("Logged successfully to DagsHub + saved locally")

2026/05/01 20:25:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 20:25:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged successfully to DagsHub + saved locally
🏃 View run angry-foal-803 at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/57f526c9a9544be8b25d65d7d393a882
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/0
